In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, get_json_object, explode, first, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType, TimestampType, IntegerType
import pyspark.sql.functions as F

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Claims")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
bronze_path = "../../data_lake/bronze/batch_data/resource_type=Claim/"
silver_claim_header_path = "../../data_lake/silver/silver_claim_header/"
silver_claim_line_item_path = "../../data_lake/silver/silver_claim_line_item/"

silver_diagnosis = "../../data_lake/silver/silver_claim_diagnosis/"
silver_procedure = "../../data_lake/silver/silver_claim_procedure/"


In [ ]:
df_bronze = spark.read.format("parquet").load(bronze_path)

In [ ]:
reference_schema = StructType([
            StructField("reference", StringType(), True),
            StructField("display", StringType(), True)
        ])

general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

type_schema = StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ])


telecom_schema = StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True),
        StructField("use", StringType(), True)
])

address_schema = StructType([
        StructField("line", ArrayType(StringType()), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postalCode", StringType(), True),
        StructField("country", StringType(), True)
    ])

identifier_schema = StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True)
    ])

extension_schema = StructType([
        StructField("url", StringType(), True),
        StructField("valueInteger", StringType(), True)
])

name_schema = StructType([
        StructField("family", StringType(), True),
        StructField("given", ArrayType(StringType()), True),
        StructField("prefix", ArrayType(StringType()), True)
])

billablePeriod_schema = StructType([
        StructField("start", TimestampType(), True),
        StructField("end", TimestampType(), True)
])

careTeam_schema = StructType([
        StructField("sequence", StringType(), True),
        StructField("provider", StringType(), True),
        StructField("role", type_schema, True),
        # StructField("qualification", type_schema, True)
])

diagnosis_schema = StructType([
        StructField("sequence", StringType(), True),
        StructField("diagnosisReference", reference_schema, True)
])

insurance_schema = StructType([
        StructField("focal", StringType(), True),
        StructField("coverage", reference_schema, True)
])

amount_schema = StructType([
        StructField("value", DoubleType(), True),
        StructField("currency", StringType(), True)
])

adjudication_schema = StructType([
        StructField("category", type_schema, True),
        StructField("amount", amount_schema, True)
])

item_schema = StructType([
        StructField("sequence", StringType(), True),
        StructField("diagnosisSequence", ArrayType(StringType()), True),
        StructField("informationSequence", ArrayType(StringType()), True),
        StructField("category", type_schema, True),
        StructField("productOrService", type_schema, True),
        StructField("servicedPeriod", billablePeriod_schema, True),
        StructField("locationCodeableConcept", type_schema, True),
        StructField("net", amount_schema, True),
        StructField("encounter", ArrayType(MapType(StringType(), StringType())), True),
        StructField("adjudication", ArrayType(adjudication_schema), True)
])

payment_schema = StructType([
        StructField("amount", amount_schema, True)
])

total_schema = StructType([
        StructField("category", type_schema, True),
        StructField("amount", amount_schema, True)
])

supportingInfo_schema = StructType([
        StructField("sequence", StringType(), True),
        StructField("category", type_schema, True),
        StructField("code", type_schema, True),
        StructField("timingDateTime", TimestampType(), True),
        StructField("valueString", StringType(), True),
        StructField("valueQuantity", amount_schema, True),
        StructField("valueReference", reference_schema, True)
])

procedure_schema = StructType([
        StructField("sequence", StringType(), True),
        StructField("procedureReference", reference_schema, True)
])

item_schema = (StructType([
    StructField("sequence",          IntegerType(),              True),
    StructField("diagnosisSequence", ArrayType(IntegerType()),   True),
    StructField("procedureSequence", ArrayType(IntegerType()),   True),
    StructField("informationSequence",ArrayType(IntegerType()),  True),
    StructField("encounter",         ArrayType(reference_schema),  True),
    StructField("productOrService", type_schema ,      True),
    StructField("net",               amount_schema,                 True),
]))


In [ ]:
df_inter = (df_bronze.select(

    col("resource.id").alias("claim_id"),
    col("resource.status").alias("status"),
    from_json(col("resource.type"), type_schema).alias("type"),
    # col("resource.subtype").alias("claim_subtype"),
    col("resource.use").alias("use"),
    F.split(from_json(col("resource.patient"), reference_schema)["reference"],":")[2].alias("patient_id"),
    from_json(col("resource.billablePeriod"), billablePeriod_schema).alias("billable_period"),
    col("resource.created").cast("timestamp").alias("created"),
    F.split(from_json(col("resource.provider"), reference_schema)["reference"],"\|")[1].alias("provider_id"),
    from_json(col("resource.priority"), type_schema).alias("priority"),
    F.split(from_json(col("resource.facility"), reference_schema)["reference"],"\|")[1].alias("facility_id"),
    from_json(col("resource.insurance"), ArrayType(insurance_schema)).alias("insurance"),
    from_json(col("resource.procedure"),ArrayType(procedure_schema)).alias("procedures"),
    from_json(col("resource.diagnosis"), ArrayType(diagnosis_schema)).alias("diagnosis"),
    from_json(col("resource.item"), ArrayType(item_schema)).alias("claim_items"),
    from_json(col("resource.total"), amount_schema).alias("total"),
    col("input_file_name").alias("input_file_name")
)
)

In [ ]:
df_claim_header = df_inter.select(
    col("claim_id"),
    col("status"),
    col("type.coding")[0]["code"].alias("claim_type"),
    col("use"),
    col("patient_id"),
    col("billable_period.start").alias("billable_period_start"),
    col("billable_period.end").alias("billable_period_end"),
    col("created"),
    col("provider_id"),
    col("priority.coding")[0]["code"].alias("priority"),
    col("facility_id"),
    col("insurance")["coverage"]["display"].alias("insurance"),
    col("total.value").alias("total_amount"),
    col("total.currency").alias("total_currency"),
    F.size("claim_items").alias("claim_count"),
    col("input_file_name"),
    current_timestamp().alias("silver_timestamp")
).drop("claim_items")


In [ ]:
df_silver_claim_line_item = df_inter.select(
    col("claim_id"),
    F.explode_outer(F.col("claim_items")).alias("item"),
    col("input_file_name")
).select(
    col("claim_id"),
    col("item.sequence").alias("item_seq"),
    F.when(F.col("item.encounter").isNotNull(),          "encounter")
     .when(F.col("item.procedureSequence").isNotNull(),  "procedure")
     .when(F.col("item.diagnosisSequence").isNotNull(),  "diagnosis")
     .when(F.col("item.informationSequence").isNotNull(),"observation")
     .otherwise("unknown")
     .alias("item_type"),
    col("item.diagnosisSequence")[0].alias("diagnosis_seq_ref"),
    col("item.procedureSequence")[0].alias("procedure_seq_ref"),
    col("item.informationSequence")[0].alias("info_seq_ref"),
    F.when(F.col("item.encounter").isNotNull(),          F.split(col("item.encounter")[0]["reference"],":")[2])
     .otherwise(None).alias("encounter_id"),
    col("item.productOrService.coding")[0]["system"].alias("service_code_system"),
    col("item.productOrService.coding")[0]["code"].alias("service_code"),
    col("item.productOrService.text").alias("service_description"),
    col("item.net.value").alias("net_amt"),
    col("item.net.currency").alias("currency"),
    col("item.net").isNotNull().alias("is_billable"),
    col("input_file_name"),
    current_timestamp().alias("silver_timestamp")
)

In [ ]:
df_diagnosis = df_inter.select(
    col("claim_id"),
    F.explode_outer(col("diagnosis")).alias("diagnosis")
).select(
    col("claim_id"),
    col("diagnosis.sequence").alias("diagnosis_seq"),
    col("diagnosis.diagnosisReference.reference").alias("diagnosis_reference"),
    current_timestamp().alias("silver_timestamp")
    #not planning to include inputfile name because it can be joined back with claim table
)

In [ ]:
df_procedure = df_inter.select(
    col("claim_id"),
    F.explode_outer(col("procedures")).alias("procedure")
).select(
    col("claim_id"),
    col("procedure.sequence").alias("procedure_seq"),
    col("procedure.procedureReference.reference").alias("procedure_reference"),
    current_timestamp().alias("silver_timestamp")
)

In [ ]:
df_claim_header.write.mode("overwrite").format("parquet").save(silver_claim_header_path)
df_silver_claim_line_item.write.mode("overwrite").format("parquet").save(silver_claim_line_item_path)

In [ ]:
df_diagnosis.write.mode("overwrite").format("parquet").save(silver_diagnosis)
df_procedure.write.mode("overwrite").format("parquet").save(silver_procedure)

In [ ]:
spark.stop()